## Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import pearsonr


In [ ]:
def load_coords(path, key_col):
    df = pd.read_excel(path)
    return df.set_index(key_col)


# ------------------------------
# Haversine для расстояния
# ------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


# Лаговая корреляция
# =========================
def max_lag_correlation(hydro, meteo, max_lag=14):
    best_corr = -1
    best_lag = 0

    for lag in range(0, max_lag + 1):
        shifted = meteo.shift(lag)
        df = pd.concat([hydro, shifted], axis=1).dropna()

        if len(df) < max_lag:
            continue

        corr, _ = pearsonr(df.iloc[:, 0], df.iloc[:, 1])
        if abs(corr) > abs(best_corr):
            best_corr = corr
            best_lag = lag

    return best_corr, best_lag


# Mutual Information
# =========================
def compute_mi(hydro, meteo):
    df = pd.concat([hydro, meteo], axis=1).dropna()
    if len(df) < 10:
        return 0
    mi = mutual_info_regression(df.iloc[:, 1:].values, df.iloc[:, 0].values)
    return float(np.mean(mi))


def rank_stations(hydro_path, meteo_folder, meteo_coords_path, hydro_coords_path, L=100):

    hydro = pd.read_excel(hydro_path, parse_dates=['Дата'])

    # координаты поста
    hydro_coords = load_coords(hydro_coords_path, 'post')
    hydro_name = hydro_coords.index[0]  # если один пост
    hydro_lat = hydro_coords.loc[hydro_name, 'lat']
    hydro_lon = hydro_coords.loc[hydro_name, 'lon']

    # координаты станций
    meteo_coords = load_coords(meteo_coords_path, 'station')

    results = []

    for file in os.listdir(meteo_folder):
        if not file.endswith(".xlsx"):
            continue

        name = file.replace(".xlsx", "")

        # проверка: есть ли координаты
        if name not in meteo_coords.index:
            print(f"Нет координат для станции {name}")
            continue

        lat = meteo_coords.loc[name, 'lat']
        lon = meteo_coords.loc[name, 'lon']

        # расстояние
        dist = haversine(hydro_lat, hydro_lon, lat, lon)

        df = pd.read_excel(os.path.join(meteo_folder, file), parse_dates=['Дата'])

        merged = pd.merge(hydro, df, on="date", how="inner")

        if merged.shape[0] < 30:
            continue

        # лучше явно выбрать признаки
        features = []
        if 'temp' in merged.columns:
            features.append('temp')
        if 'precip' in merged.columns:
            features.append('precip')

        if not features:
            continue

        meteo = merged[features].mean(axis=1)

        corr, lag = max_lag_correlation(merged['level'], meteo)
        mi = compute_mi(merged['level'], meteo)

        # вес расстояния
        distance_weight = np.exp(-dist / L)

        score = (0.5 * abs(corr) + 0.5 * mi) * distance_weight

        results.append({
            "station": name,
            "distance_km": dist,
            "corr": corr,
            "lag": lag,
            "mi": mi,
            "weight": distance_weight,
            "score": score
        })

    df = pd.DataFrame(results)
    df = df.sort_values("score", ascending=False)

    return df


df = rank_stations(
    hydro_path="https://github.com/JujikMilkivey/datasets_for_forecasting/blob/main/Data-structured/%D0%97%D0%B0%D0%BF%D0%B0%D0%B4%D0%BD%D0%BE-%D0%9A%D0%B0%D1%81%D0%BF%D0%B8%D0%B9%D1%81%D0%BA%D0%B8%D0%B9/84108.xlsx",
    meteo_folder="https://github.com/JujikMilkivey/datasets_for_forecasting/tree/main/Data-structured/Meteo",
    meteo_coords_path="https://github.com/JujikMilkivey/datasets_for_forecasting/blob/main/Data-structured/meteo_stations_coords.xlsx",
    hydro_coords_path="https://github.com/JujikMilkivey/datasets_for_forecasting/blob/main/Data-structured/hydro_stations_coords.xlsx"
)

# df.to_excel("results/station_ranking.csv", index=False)

ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [ ]:
# ------------------------------
# Haversine для расстояния
# ------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# ------------------------------
# Подготовка признаков для горного бассейна
# ------------------------------
def prepare_mountain_features(meteo_df):
    df = meteo_df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')

    # 7-дневные осадки
    df['P7'] = df['Precipitation'].rolling(7).sum()

    # положительные температуры
    df['T_pos'] = df['Avg_temp'].clip(lower=0)

    # сумма положительных температур (накопление)
    df['Degree_days'] = df['T_pos'].rolling(10).sum()

    # сглаженный снег
    if 'Высота снежного покрова' in df.columns:
        df['Snow_smooth'] = df['Высота снежного покрова'].rolling(5).mean()

    return df

# ------------------------------
# Оценка одной станции (без координат в meteo_df)
# ------------------------------
def score_mountain_station(hydro_df, meteo_df, max_lag=15):
    hydro = hydro_df.copy()
    hydro['Дата'] = pd.to_datetime(hydro['Дата'])
    hydro = hydro.set_index('Дата')['Расход']

    meteo_df = prepare_mountain_features(meteo_df)

    features = {
        'P7': 0.25,
        'Degree_days': 0.35,
        'Snow_smooth': 0.25
    }

    total_score = 0
    feature_details = {}

    for feature, weight in features.items():
        if feature not in meteo_df.columns:
            continue
        meteo = meteo_df.set_index('Date')[feature]
        combined = pd.concat([hydro, meteo], axis=1).dropna()
        combined.columns = ['Q', 'X']

        best_corr = 0
        best_lag = 0
        for lag in range(max_lag + 1):
            corr = combined['Q'].corr(combined['X'].shift(lag))
            if abs(corr) > abs(best_corr):
                best_corr = corr
                best_lag = lag

        feature_score = abs(best_corr)
        total_score += weight * feature_score
        feature_details[feature] = {'corr': best_corr, 'lag': best_lag}

    return total_score, feature_details

# ------------------------------
# Подбор всех станций из папки с coords_df
# ------------------------------
def select_mountain_stations(hydro_df, meteo_folder, hydro_coords, coords_df, max_lag=15):
    results = []
    meteo_folder = Path(meteo_folder)

    for file in meteo_folder.glob('*.xlsx'):
        station_name = file.stem
        meteo_df = pd.read_excel(file)

        # координаты берём из coords_df
        row = coords_df[coords_df['Станция'] == station_name]
        if row.empty:
            print(f'Нет координат для {station_name}')
            continue

        lat_station = row['Широта'].values[0]
        lon_station = row['Долгота'].values[0]

        score, details = score_mountain_station(hydro_df, meteo_df, max_lag=max_lag)

        distance = haversine(hydro_coords[0], hydro_coords[1], lat_station, lon_station)
        dist_score = 1 / (1 + distance)

        # добавляем вклад расстояния (15%)
        final_score = score + 0.15 * dist_score

        results.append({
            'Станция': station_name,
            'Итоговый балл': final_score,
            'Расстояние (км)': distance,
            'P7_corr': details.get('P7', {}).get('corr'),
            'Degree_days_corr': details.get('Degree_days', {}).get('corr'),
            'Snow_corr': details.get('Snow_smooth', {}).get('corr'),
        })

    result_df = pd.DataFrame(results)
    return result_df.sort_values('Итоговый балл', ascending=False)

In [ ]:
coords_df_path = "C:/Users/UserHome/Desktop/Аспирантура/Forecasting_2025/Data-structured/stations_coords.xlsx"
coords_df = pd.read_excel(win_to_wsl_path(coords_df_path))    
hydro_coords = (43.47, 46.33)

ranking = select_mountain_stations(
    hydro_df,
    win_to_wsl_path(r"C:\Users\UserHome\Desktop\Аспирантура\Forecasting_2025\Data-structured\Meteo"),
    hydro_coords,
    coords_df
)